# 🔍 Phase 1: Data Exploration
**Goal**: Understand what's in our candidate data before we build anything.

We use `sample_candidates.json` (50 candidates) as our playground.
The full dataset (`candidates.jsonl`) has 100,000 — same structure, just bigger.

---
**Run each cell one by one. Read the output carefully.**

## Cell 1 — Load the data

**What**: Load the 50-candidate sample into Python.

**Why**: We need to see the raw structure before flattening it into a DataFrame.

**What to look for**: Is `samples` a list? What keys does the first candidate have?

In [1]:
import json
import pandas as pd

with open('sample_candidates.json') as f:
    samples = json.load(f)

print(f"Number of candidates: {len(samples)}")
print(f"Type of samples: {type(samples)}")
print(f"\nTop-level keys in each candidate:")
print(list(samples[0].keys()))

Number of candidates: 50
Type of samples: <class 'list'>

Top-level keys in each candidate:
['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals']


## Cell 2 — Look at ONE candidate in full

**What**: Print the first candidate's full profile.

**Why**: Before writing any code, you need to *read* the data like a human recruiter would.

**Question to answer after running this**: Is this candidate good for a Senior AI Engineer role? Why or why not?

In [2]:
# Let's print candidate #1 nicely
c = samples[0]

print("=" * 60)
print(f"ID       : {c['candidate_id']}")
print(f"Title    : {c['profile']['current_title']}")
print(f"YOE      : {c['profile']['years_of_experience']} years")
print(f"Summary  : {c['profile']['summary'][:300]}...")

print("\n--- SKILLS ---")
for s in c['skills']:
    print(f"  {s['name']:30s} | {s['proficiency']:12s} | endorsements: {s['endorsements']} | months: {s['duration_months']}")

print("\n--- BEHAVIORAL SIGNALS ---")
sig = c['redrob_signals']
print(f"  open_to_work          : {sig['open_to_work_flag']}")
print(f"  last_active_date      : {sig['last_active_date']}")
print(f"  github_activity_score : {sig['github_activity_score']}")
print(f"  recruiter_response    : {sig['recruiter_response_rate']}")
print(f"  interview_completion  : {sig['interview_completion_rate']}")
print(f"  offer_acceptance      : {sig['offer_acceptance_rate']}")
print(f"  skill_assessments     : {sig['skill_assessment_scores']}")

ID       : CAND_0000001
Title    : Backend Engineer
YOE      : 6.9 years
Summary  : Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — ...

--- SKILLS ---
  Tailwind                       | intermediate | endorsements: 3 | months: 13
  NLP                            | advanced     | endorsements: 37 | months: 26
  Image Classification           | advanced     | endorsements: 7 | months: 40
  Fine-tuning LLMs               | advanced     | endorsements: 21 | months: 36
  Weights & Biases               | intermediate | endorsements: 13 | months: 30
  Speech Recognition             | advanced     | endorsements: 52 | months: 33
  Photoshop                      | intermediate | endorsements: 8 | months: 24
  TTS                            | adva

## Cell 3 — Flatten into a Pandas DataFrame

**What**: Turn the nested JSON into a flat table — one row per candidate.

**Why**: We can't run `.sort_values()` or `.describe()` on raw JSON. Pandas is our analysis tool.

**Key concept**: We're pulling out the fields we care about. Nested fields (like `redrob_signals`) need to be unpacked manually.

In [3]:
rows = []

for c in samples:
    profile = c['profile']
    sig = c['redrob_signals']
    
    row = {
        # Identity
        'candidate_id'           : c['candidate_id'],
        'current_title'          : profile['current_title'],
        'years_of_experience'    : profile['years_of_experience'],
        'headline'               : profile.get('headline', ''),
        'summary'                : profile.get('summary', ''),
        
        # Behavioral signals
        'open_to_work'           : sig['open_to_work_flag'],
        'last_active_date'       : sig['last_active_date'],
        'github_score'           : sig['github_activity_score'],
        'recruiter_response_rate': sig['recruiter_response_rate'],
        'interview_completion'   : sig['interview_completion_rate'],
        'offer_acceptance'       : sig['offer_acceptance_rate'],
        'profile_completeness'   : sig['profile_completeness_score'],
        'notice_period_days'     : sig['notice_period_days'],
        
        # Raw lists (we'll process these next)
        'skills'                 : c['skills'],
        'career_history'         : c['career_history'],
        'certifications'         : c['certifications'],
    }
    rows.append(row)

df = pd.DataFrame(rows)

print(f"DataFrame shape: {df.shape}")
print(f"\nColumn names:")
print(df.columns.tolist())


df.head(5)

DataFrame shape: (50, 16)

Column names:
['candidate_id', 'current_title', 'years_of_experience', 'headline', 'summary', 'open_to_work', 'last_active_date', 'github_score', 'recruiter_response_rate', 'interview_completion', 'offer_acceptance', 'profile_completeness', 'notice_period_days', 'skills', 'career_history', 'certifications']


,candidate_id,current_title,years_of_experience,headline,summary,open_to_work,last_active_date,github_score,recruiter_response_rate,interview_completion,offer_acceptance,profile_completeness,notice_period_days,skills,career_history,certifications
0,CAND_0000001,Backend Engineer,6.9,"Backend Engineer | SQL, Spark, Cloud",Software / data professional with 6.9 years of...,True,2026-05-20,9.2,0.34,0.71,0.58,86.9,60,"[{'name': 'Tailwind', 'proficiency': 'intermed...","[{'company': 'Mindtree', 'title': 'Backend Eng...",[]
1,CAND_0000002,Operations Manager,12.5,Operations Manager | 12.5+ yrs experience,Professional with 12.5+ years of experience. M...,True,2025-11-12,-1.0,0.29,0.62,-1.00,78.7,60,"[{'name': 'Project Management', 'proficiency':...","[{'company': 'Wipro', 'title': 'Operations Man...",[]
2,CAND_0000003,Customer Support,1.1,Customer Support | 1.1+ yrs experience,Professional with 1.1+ years of experience. I'...,False,2026-03-21,-1.0,0.46,0.86,-1.00,31.9,150,"[{'name': 'Angular', 'proficiency': 'intermedi...","[{'company': 'TCS', 'title': 'Customer Support...",[]
3,CAND_0000004,Marketing Manager,3.8,Marketing Manager | Driving business outcomes,Professional with 3.8+ years of experience. My...,False,2026-03-25,-1.0,0.26,0.35,-1.00,28.5,120,"[{'name': 'Node.js', 'proficiency': 'intermedi...","[{'company': 'Dunder Mifflin', 'title': 'Marke...","[{'name': 'AWS Certified Cloud Practitioner', ..."
4,CAND_0000005,Accountant,11.0,Accountant | Helping teams scale,Professional with 11.0+ years of experience. I...,True,2025-10-01,-1.0,0.37,0.74,-1.00,84.6,30,"[{'name': 'SQL', 'proficiency': 'beginner', 'e...","[{'company': 'Stark Industries', 'title': 'Acc...",[]


## Cell 4 — Distribution of job titles

**What**: What roles do these 50 candidates currently hold?

**Why**: Most candidates in a 100K pool are NOT AI engineers. We need to understand the noise.

**Look for**: Are there obvious non-relevant candidates (e.g., Accountants, Operations Managers)? These are honeypot candidates or irrelevant profiles.

In [4]:
print("Current Titles (sorted by frequency):")
print(df['current_title'].value_counts())

print("\nYears of Experience Stats:")
print(df['years_of_experience'].describe())

Current Titles (sorted by frequency):
current_title
Operations Manager                 5
Business Analyst                   5
Mechanical Engineer                5
Project Manager                    5
Frontend Engineer                  4
Marketing Manager                  3
Accountant                         3
Customer Support                   2
Civil Engineer                     2
Software Engineer                  2
HR Manager                         2
Graphic Designer                   2
Backend Engineer                   1
Data Engineer                      1
QA Engineer                        1
DevOps Engineer                    1
Recommendation Systems Engineer    1
.NET Developer                     1
Full Stack Developer               1
Java Developer                     1
Cloud Engineer                     1
Mobile Developer                   1
Name: count, dtype: int64

Years of Experience Stats:
count    50.000000
mean      7.070000
std       3.956614
min       1.100000
25% 

## Cell 5 — Who's actually active?

**What**: Check open_to_work status and when they were last active.

**Why**: A brilliant candidate who logged in 14 months ago and isn't open to work is a dead lead.

**Key insight**: The hackathon *intentionally* includes behavioral decoys — great skills, but inactive.

In [5]:
from datetime import datetime, date

# Convert last_active_date to a datetime
df['last_active_date'] = pd.to_datetime(df['last_active_date'])

# How many days since last active (reference = June 26, 2026)
reference_date = pd.Timestamp('2026-06-26')
df['days_since_active'] = (reference_date - df['last_active_date']).dt.days

print("open_to_work breakdown:")
print(df['open_to_work'].value_counts())

print("\nDays since last active:")
print(df['days_since_active'].describe())

print("\nCandidates inactive for > 180 days:")
print(df[df['days_since_active'] > 180][['candidate_id', 'current_title', 'days_since_active', 'open_to_work']])

open_to_work breakdown:
open_to_work
False    34
True     16
Name: count, dtype: int64

Days since last active:
count     50.00000
mean     140.50000
std       74.09598
min       32.00000
25%       82.50000
50%      131.00000
75%      196.75000
max      270.00000
Name: days_since_active, dtype: float64

Candidates inactive for > 180 days:
    candidate_id        current_title  days_since_active  open_to_work
1   CAND_0000002   Operations Manager                226          True
4   CAND_0000005           Accountant                268          True
7   CAND_0000008   Operations Manager                195         False
11  CAND_0000012   Operations Manager                241         False
15  CAND_0000016           Accountant                187          True
16  CAND_0000017           Accountant                234         False
19  CAND_0000020  Mechanical Engineer                264         False
20  CAND_0000021      Project Manager                217         False
25  CAND_0000026    

## Cell 6 — Skill analysis

**What**: Build a list of all AI/ML skills mentioned across all candidates.

**Why**: We need to define what "relevant skills" means for a Senior AI Engineer before we score.

**Think about**: What skills would you look for if YOU were hiring a Senior AI Engineer?

In [6]:
from collections import Counter

# Flatten all skills across all candidates
all_skills = []
for skills_list in df['skills']:
    for s in skills_list:
        all_skills.append(s['name'])

skill_counts = Counter(all_skills)

print(f"Total unique skills across all 50 candidates: {len(skill_counts)}")
print("\nTop 30 most common skills:")
for skill, count in skill_counts.most_common(30):
    print(f"  {skill:35s} : {count}")

Total unique skills across all 50 candidates: 109

Top 30 most common skills:
  AWS                                 : 10
  Data Pipelines                      : 10
  Node.js                             : 9
  Sales                               : 9
  Figma                               : 9
  Tailwind                            : 8
  GCP                                 : 8
  Content Writing                     : 8
  GraphQL                             : 8
  gRPC                                : 8
  Terraform                           : 8
  Scrum                               : 8
  Java                                : 8
  Spring Boot                         : 8
  Hadoop                              : 8
  ETL                                 : 8
  Weights & Biases                    : 7
  React                               : 7
  TypeScript                          : 7
  Kubernetes                          : 7
  Redux                               : 7
  Airflow                             

## Cell 7 — Spot a honeypot

**What**: Find candidates with suspiciously perfect skill profiles.

**Why**: A honeypot is a fake "too good to be true" profile. If someone lists 20 skills, all at 'expert' level, all with 0 endorsements — that's suspicious.

**Key signal**: Proficiency claims vs. actual endorsements vs. assessment scores.

In [7]:
# For each candidate, count how many skills they claim 'expert' but have 0 endorsements
def count_suspicious_skills(skills_list):
    suspicious = 0
    for s in skills_list:
        if s['proficiency'] in ['expert', 'advanced'] and s['endorsements'] == 0:
            suspicious += 1
    return suspicious

df['suspicious_skill_count'] = df['skills'].apply(count_suspicious_skills)
df['total_skill_count'] = df['skills'].apply(len)

print("Candidates with most suspicious skills (high proficiency, 0 endorsements):")
suspicious_df = df[['candidate_id', 'current_title', 'total_skill_count', 'suspicious_skill_count']]
print(suspicious_df.sort_values('suspicious_skill_count', ascending=False).head(10))

Candidates with most suspicious skills (high proficiency, 0 endorsements):
    candidate_id                    current_title  total_skill_count  \
42  CAND_0000043                   Cloud Engineer                 17   
9   CAND_0000010                    Data Engineer                 17   
0   CAND_0000001                 Backend Engineer                 17   
37  CAND_0000038                   Java Developer                 11   
28  CAND_0000029                 Business Analyst                  8   
29  CAND_0000030                Marketing Manager                  9   
30  CAND_0000031  Recommendation Systems Engineer                 17   
31  CAND_0000032                   .NET Developer                 10   
32  CAND_0000033                 Graphic Designer                 10   
33  CAND_0000034                 Business Analyst                 10   

    suspicious_skill_count  
42                       1  
9                        1  
0                        0  
37              

## Cell 8 — Quick visual of behavioral signals

**What**: Print a summary table of all behavioral signals side by side.

**Why**: This is what will separate our solution from basic keyword matchers. Real recruiter platforms use all of these.

In [8]:
behavioral_cols = [
    'candidate_id', 'current_title',
    'open_to_work', 'days_since_active',
    'github_score', 'recruiter_response_rate',
    'interview_completion', 'offer_acceptance'
]

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print("Behavioral signals summary (sorted by github_score):")
print(df[behavioral_cols].sort_values('github_score', ascending=False).to_string(index=False))

Behavioral signals summary (sorted by github_score):
candidate_id                   current_title  open_to_work  days_since_active  github_score  recruiter_response_rate  interview_completion  offer_acceptance
CAND_0000023               Software Engineer         False                 81          48.5                     0.57                  0.90             -1.00
CAND_0000024                      HR Manager         False                157          46.3                     0.78                  0.72             -1.00
CAND_0000050                Business Analyst         False                247          44.7                     0.42                  0.58             -1.00
CAND_0000016                      Accountant          True                187          42.9                     0.64                  0.66             -1.00
CAND_0000029                Business Analyst         False                270          42.5                     0.12                  0.67             -1.00
CAND_

---
## ✅ Phase 1 Complete — What You Should Know Now

After running all cells, you should be able to answer:

1. What does a single candidate record look like? (nested JSON → flat DataFrame)
2. What fraction of candidates are NOT relevant (wrong job titles)?
3. What behavioral signals exist and what do they mean?
4. What does a 'suspicious' (honeypot) profile look like?
5. What skills are most common in this pool?

---
**Next**: Tell me what you noticed! Then we'll design our scoring formula together. 🚀